# ByT5 + Philologist Two-Tier Ensemble (CPU) v9

Weight-blended **5-model** ByT5 ensemble + **Flan-T5 Philologist** refinement.
TF-IDF retrieval fallback with chrF++ consensus scoring.
Runs on **CPU only** - no GPU required.

**v9 improvements:**
1. **Flan-T5 Philologist** refinement tier (250M params, trained to refine ByT5 output)
2. Two-tier pipeline: ByT5 → consensus select → Philologist → final output
3. Multi-beam consensus scoring with TR-TRY cross-validation
4. 5-model ByT5 weight-blended ensemble with RAG-augmented ByT5
5. ORACC corpus (2,117+ pairs) + translation memory exact match
6. OA Lexicon proper noun normalization (4,014 PN mappings)

In [ ]:
import os, re, time, math, unicodedata, gc
from collections import Counter
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: cpu (forced)")

# Debug: check dataset mounts (2 levels deep)
print("\n=== /kaggle/input/ contents ===")
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    for item in sorted(os.listdir(input_dir)):
        full = os.path.join(input_dir, item)
        if os.path.isdir(full):
            sub_items = sorted(os.listdir(full))
            print(f"  {item}/ ({len(sub_items)} items)")
            for f in sub_items[:8]:
                fpath = os.path.join(full, f)
                if os.path.isdir(fpath):
                    sub2 = sorted(os.listdir(fpath))
                    print(f"    {f}/ ({len(sub2)} items)")
                    for f2 in sub2[:5]:
                        fpath2 = os.path.join(fpath, f2)
                        if os.path.isdir(fpath2):
                            sub3 = sorted(os.listdir(fpath2))
                            print(f"      {f2}/ ({len(sub3)} items): {sub3[:5]}")
                        else:
                            print(f"      {f2} ({os.path.getsize(fpath2)})")
                else:
                    print(f"    {f} ({os.path.getsize(fpath)})")
        else:
            print(f"  {item}")
print("=" * 40)

## Configuration

In [ ]:
# Competition data path - check both mount points
DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
LEXICON_PATH = f"{DATA_DIR}/OA_Lexicon_eBL.csv"

print(f"DATA_DIR: {DATA_DIR}")
print(f"TRAIN_PATH exists: {os.path.exists(TRAIN_PATH)}")
print(f"TEST_PATH exists: {os.path.exists(TEST_PATH)}")
print(f"LEXICON_PATH exists: {os.path.exists(LEXICON_PATH)}")

# ORACC supplementary corpus path
def find_oracc_path():
    """Find ORACC parallel corpus dataset on Kaggle."""
    candidates = [
        "/kaggle/input/oracc-akkadian-english-parallel-corpus",
        "/kaggle/input/datasets/manwithacat/oracc-akkadian-english-parallel-corpus",
    ]
    for base in candidates:
        if os.path.exists(base):
            for f in sorted(os.listdir(base)):
                if f.endswith('.csv'):
                    return os.path.join(base, f)
            for root, dirs, files in os.walk(base):
                for f in files:
                    if f.endswith('.csv'):
                        return os.path.join(root, f)
    return None

ORACC_PATH = find_oracc_path()
print(f"ORACC_PATH: {ORACC_PATH} [{'OK' if ORACC_PATH else 'not found'}]")

# Model paths - resolve with fallback for different Kaggle mount structures
def find_model_path(dataset_slug, subpaths):
    """Find model directory trying multiple Kaggle mount patterns."""
    owner, name = dataset_slug.split("/")
    base_dirs = [
        f"/kaggle/input/{name}",
        f"/kaggle/input/datasets/{owner}/{name}",
        f"/kaggle/input/datasets/{owner}",
    ]
    for base in base_dirs:
        if not os.path.exists(base):
            continue
        for sub in subpaths:
            p = f"{base}/{sub}" if sub else base
            if os.path.exists(p) and os.path.isfile(os.path.join(p, "config.json")):
                return p
        for root, dirs, files in os.walk(base):
            if "config.json" in files:
                return root
    return f"/kaggle/input/{name}"

# 5-model ByT5 ensemble: 4 original + RAG-augmented model
MODEL_CONFIGS = [
    ("jeanjean111/byt5-base-big-data2", ["", "byt5-base-big-data2"], 0.995),
    ("llkh0a/byt5-akkadian-model", ["", "byt5-akkadian-model"], 0.985),
    ("qifeihhh666/train-gap-all-2", [
        "byt5-base-akkadian_gap_setence2",
        "train_GAP_all_2/byt5-base-akkadian_gap_setence2", "",
    ], 0.39),
    ("assiaben/final-byt5", [
        "byt5-akkadian-optimized-34x",
        "", "final-byt5",
    ], 0.99),
    ("manwithacat/byt5-rag-akkadian-v1", [
        "", "byt5-rag-akkadian-v1",
    ], 0.85),
]

MODEL_PATHS = [find_model_path(slug, subs) for slug, subs, _ in MODEL_CONFIGS]
WEIGHTS = [w for _, _, w in MODEL_CONFIGS]

for i, (p, w) in enumerate(zip(MODEL_PATHS, WEIGHTS)):
    exists = os.path.exists(os.path.join(p, "config.json"))
    print(f"ByT5 Model {i} ({w}): {p} [{'OK' if exists else 'MISSING'}]")

# Use the first available model as base for tokenizer
TOKENIZER_PATH = next(p for p in MODEL_PATHS if os.path.exists(os.path.join(p, "config.json")))

# Flan-T5 Philologist refinement model (Tier 2)
# Trained to refine raw ByT5 translations to gold reference style
PHILOLOGIST_CANDIDATES = [
    "/kaggle/input/akkadian-flan-t5-philologist-v1-0-0/pytorch/transformers/1",
    "/kaggle/input/akkadian-flan-t5-philologist-v1-0-0/pytorch/transformers",
    "/kaggle/input/akkadian-flan-t5-philologist-v1-0-0",
]
PHILOLOGIST_PATH = None
for p in PHILOLOGIST_CANDIDATES:
    if os.path.exists(p) and os.path.isfile(os.path.join(p, "config.json")):
        PHILOLOGIST_PATH = p
        break
    elif os.path.exists(p):
        for root, dirs, files in os.walk(p):
            if "config.json" in files:
                PHILOLOGIST_PATH = root
                break
        if PHILOLOGIST_PATH:
            break

print(f"Philologist: {PHILOLOGIST_PATH} [{'OK' if PHILOLOGIST_PATH else 'NOT FOUND'}]")

PHILOLOGIST_PREFIX = "Refine Akkadian translation: "
PHILOLOGIST_GEN_PARAMS = {
    'num_beams': 4,
    'max_new_tokens': 384,
    'length_penalty': 1.0,
    'early_stopping': True,
}

# Inference parameters
DEVICE = torch.device("cpu")
MAX_LEN = 496
TASK_PREFIX = "translate Akkadian to English: "
TIME_BUDGET_MINUTES = 400

# Multi-candidate consensus parameters
NUM_RETURN_SEQUENCES = 4  # Generate top N beam candidates
CONSENSUS_TRTRY_WEIGHT = 0.3  # Weight for TR-TRY agreement in consensus

# TR-TRY parameters
W_CHAR = 0.75
W_WORD = 0.20
W_SEQ = 0.05
TOP_K = 60
RERANK_K = 10
LEN_PENALTY_POWER = 0.3
MIN_ACCEPT_SCORE = 0.10

# Hybrid selection thresholds
TR_TRY_HIGH_CONF = 0.70
BYT5_GARBAGE_RATIO = 0.5

# Sentence alignment threshold (chars)
LONG_TEXT_THRESHOLD = 256

## Preprocessing

In [ ]:
_SUBSCRIPT_TABLE = str.maketrans("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089", "0123456789")
_SUPERSCRIPT_TABLE = str.maketrans("\u2070\u00b9\u00b2\u00b3\u2074\u2075\u2076\u2077\u2078\u2079", "0123456789")
_DET_PATTERN = re.compile(
    r"\{(?:d|f|m|ki|kur|uru|lu2?|na4|gi[s\u0161]|mul|u[d\u0161]|an|i[d\u0161]|tug2?|ku[s\u0161])\}",
    re.IGNORECASE,
)


def preprocess_input(text):
    """Clean Akkadian transliteration for ByT5 input."""
    if not isinstance(text, str) or not text.strip():
        return TASK_PREFIX

    t = str(text)

    # Normalize gaps
    t = re.sub(r'\[\s*\.\s*\.\s*\.\s*\]', '<big_gap>', t)
    t = re.sub(r'\(\s*\.\s*\.\s*\.\s*\)', '<big_gap>', t)
    t = re.sub(r'\.{3,}', '<big_gap>', t)
    t = re.sub(r'\u2026{2,}', '<big_gap>', t)
    t = re.sub(r'\u2026', '<big_gap>', t)
    t = re.sub(r'\[x+\]', '<gap>', t)
    t = re.sub(r'xx+', '<gap>', t)
    t = re.sub(r'\s+x\s+', ' <gap> ', t)
    t = re.sub(r'<gap>(\s*<gap>)+', '<gap>', t)
    t = re.sub(r'<big_gap>(\s*<big_gap>)+', '<big_gap>', t)

    # Normalize determinatives
    t = _DET_PATTERN.sub('', t)

    # Normalize subscripts and superscripts
    t = t.translate(_SUBSCRIPT_TABLE)
    t = t.translate(_SUPERSCRIPT_TABLE)

    # Remove scholarly symbols
    t = t.translate(str.maketrans('', '', '\u2308\u2309\u230a\u230b\u00b0'))

    # Collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip()

    # Truncate
    if len(t) > 800:
        t = t[:800] + ' <big_gap>'

    return TASK_PREFIX + t


print("Preprocessing ready.")
print(f"  Example: {preprocess_input('KI\u0160IB {d}UTU-ba-ni DUMU e-na-su-en\u2082')[:80]}...")

## Postprocessing

In [ ]:
_BAD_OUTPUT_CHARS = '!?()\"\u2014\u2013<>\u2308\u230b\u230a[]+\u02be/;'
_BRACKET_PATTERN = re.compile(r'[\(\[][^\)\]]*[\)\]]')
_GRAMMAR_PATTERN = re.compile(
    r'\((?:fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)', re.IGNORECASE
)

_FRACTION_MAP = {
    '1/2': '\u00bd', '1/3': '\u2153', '2/3': '\u2154',
    '1/4': '\u00bc', '3/4': '\u00be', '1/5': '\u2155',
    '1/6': '\u2159', '5/6': '\u215a', '1/8': '\u215b',
}

_SHORT_INPUT_MAP = {
    'a-na': 'To', 'um-ma': 'saying:', 'IGI': 'Witnesses:',
    'KI\u0160IB': 'Seal of', 'ma-na': 'mina of silver',
    '\u0161u-ma': 'If he does not pay', 'i-na': 'In',
    'ITU.KAM': 'Month:', 'i\u0161-t\u00f9': 'From the',
    'K\u00d9.BABBAR': 'silver', 'G\u00cdN': 'shekels of silver',
    '\u00fa-\u1e63a-\u00e1b': 'he will add interest',
    'li-mu-um': 'Eponymy of', '\u00f9': 'Also,',
    '\u0161a': 'of', 'l\u00e1': '', 'URUDU': 'copper',
    'x': '', '\u2026': '', 'a-ma-kam': 'Here',
    'en-um-a-\u0161ur': 'Ennum-A\u0161\u0161ur',
    'k\u00e0-ru-um': 'The colony',
}


def _remove_ngram_loops(text, min_n=3, max_n=8):
    """Remove looping n-gram patterns."""
    words = text.split()
    if len(words) < min_n * 2:
        return text
    for n in range(max_n, min_n - 1, -1):
        i = 0
        cleaned = []
        while i < len(words):
            if i + 2 * n <= len(words):
                chunk = words[i:i + n]
                next_chunk = words[i + n:i + 2 * n]
                if chunk == next_chunk:
                    cleaned.extend(chunk)
                    i += n
                    while i + n <= len(words) and words[i:i + n] == chunk:
                        i += n
                    continue
            cleaned.append(words[i])
            i += 1
        words = cleaned
    return ' '.join(words)


def postprocess_translation(text):
    """Full postprocessing pipeline for model output."""
    if not isinstance(text, str) or not text.strip():
        return ''

    t = text

    # Transliterate special chars
    t = t.replace('\u1e2b', 'h').replace('\u1e2a', 'H')
    t = t.translate(_SUBSCRIPT_TABLE)
    t = t.translate(_SUPERSCRIPT_TABLE)

    # Normalize gap tokens in output
    t = re.sub(r'\[x\]|\(x\)|\bx\b', '<gap>', t, flags=re.IGNORECASE)
    t = re.sub(r'\.{3,}|\u2026|\[\.+\]', '<big_gap>', t)
    t = re.sub(r'<gap>\s*<gap>', ' <big_gap> ', t)
    t = re.sub(r'<big_gap>\s*<big_gap>', ' <big_gap> ', t)

    # Remove grammar annotations
    t = _GRAMMAR_PATTERN.sub('', t)

    # Remove bracket content (improves LB)
    t = t.replace('<gap>', '\x00GAP\x00').replace('<big_gap>', '\x00BIG\x00')
    t = _BRACKET_PATTERN.sub('', t)
    t = t.replace('\x00GAP\x00', '<gap>').replace('\x00BIG\x00', '<big_gap>')

    # Remove special chars
    t = t.replace('<gap>', '\x00GAP\x00').replace('<big_gap>', '\x00BIG\x00')
    t = t.translate(str.maketrans('', '', _BAD_OUTPUT_CHARS))
    t = t.replace('\x00GAP\x00', ' <gap> ').replace('\x00BIG\x00', ' <big_gap> ')

    # Convert fractions
    for frac, symbol in _FRACTION_MAP.items():
        t = t.replace(frac, symbol)

    # Remove repeated words/phrases
    t = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', t)
    for n in range(4, 1, -1):
        pat = r'\b((?:\w+\s+){' + str(n - 1) + r'}\w+)(?:\s+\1\b)+'
        t = re.sub(pat, r'\1', t)
    t = _remove_ngram_loops(t)

    # Remove big_gap tokens
    t = ' '.join(t.replace('<big_gap>', '').split())

    # Truncate to complete sentence if long
    if len(t) > 200:
        last_period = t.rfind('.')
        last_comma = t.rfind(',')
        cut = max(last_period, last_comma)
        if cut > len(t) * 0.5:
            t = t[:cut].rstrip()

    # Final cleanup
    t = re.sub(r'\s+', ' ', t).strip().strip('-').strip()
    return t


def get_short_translation(raw_input):
    """Return dictionary translation for 1-token inputs."""
    tokens = str(raw_input).strip().split()
    if len(tokens) == 1:
        return _SHORT_INPUT_MAP.get(tokens[0])
    return None


def is_garbage_output(text):
    """Detect if model output is garbage (repetitive/empty)."""
    if not text or len(text.strip()) < 3:
        return True
    words = text.split()
    if len(words) < 2:
        return False
    # Check word-level repetition ratio
    unique = len(set(words))
    ratio = unique / len(words)
    if ratio < 0.3 and len(words) > 5:
        return True
    return False


print("Postprocessing ready.")

## chrF++ Consensus Scoring

Generate multiple beam candidates and select the best by chrF++ consensus with TR-TRY cross-validation.

In [ ]:
def compute_chrf_score(hypothesis, reference, char_order=6, word_order=2, beta=2.0):
    """Compute chrF++ score (character + word n-gram F-score)."""
    if not hypothesis or not reference:
        return 0.0

    def get_ngrams(text, n, use_chars=True):
        ngrams = Counter()
        if use_chars:
            for i in range(len(text) - n + 1):
                ngrams[text[i:i + n]] += 1
        else:
            words = text.split()
            for i in range(len(words) - n + 1):
                ngrams[tuple(words[i:i + n])] += 1
        return ngrams

    total_precision = 0.0
    total_recall = 0.0
    count = 0

    # Character n-grams
    for n in range(1, char_order + 1):
        hyp_ng = get_ngrams(hypothesis, n, use_chars=True)
        ref_ng = get_ngrams(reference, n, use_chars=True)
        matches = sum(min(hyp_ng[ng], ref_ng[ng]) for ng in hyp_ng if ng in ref_ng)
        hyp_total = sum(hyp_ng.values())
        ref_total = sum(ref_ng.values())
        if hyp_total > 0:
            total_precision += matches / hyp_total
        if ref_total > 0:
            total_recall += matches / ref_total
        count += 1

    # Word n-grams (the "++" part)
    for n in range(1, word_order + 1):
        hyp_ng = get_ngrams(hypothesis, n, use_chars=False)
        ref_ng = get_ngrams(reference, n, use_chars=False)
        matches = sum(min(hyp_ng[ng], ref_ng[ng]) for ng in hyp_ng if ng in ref_ng)
        hyp_total = sum(hyp_ng.values())
        ref_total = sum(ref_ng.values())
        if hyp_total > 0:
            total_precision += matches / hyp_total
        if ref_total > 0:
            total_recall += matches / ref_total
        count += 1

    avg_p = total_precision / max(count, 1)
    avg_r = total_recall / max(count, 1)

    if avg_p + avg_r == 0:
        return 0.0

    beta_sq = beta ** 2
    return (1 + beta_sq) * avg_p * avg_r / (beta_sq * avg_p + avg_r)


def consensus_select(candidates, external_refs=None, external_weight=0.3):
    """Select best candidate by chrF++ consensus with optional external references.

    Args:
        candidates: List of translation strings from beam search
        external_refs: Optional list of (translation, weight) tuples from other systems
        external_weight: Weight for external reference agreement (0-1)

    Returns:
        Best candidate string
    """
    if not candidates:
        return ''
    if len(candidates) == 1:
        return candidates[0]

    # Remove exact duplicates while preserving order
    seen = set()
    unique_cands = []
    for c in candidates:
        c_clean = c.strip()
        if c_clean and c_clean not in seen:
            seen.add(c_clean)
            unique_cands.append(c_clean)

    if not unique_cands:
        return candidates[0]
    if len(unique_cands) == 1:
        return unique_cands[0]

    best_score = -1
    best_cand = unique_cands[0]

    for i, cand in enumerate(unique_cands):
        # Internal consensus: average chrF++ against other candidates
        internal_score = 0.0
        for j, other in enumerate(unique_cands):
            if i != j:
                internal_score += compute_chrf_score(cand, other)
        internal_score /= max(len(unique_cands) - 1, 1)

        # External agreement: chrF++ against TR-TRY or other references
        external_score = 0.0
        if external_refs:
            total_ext_weight = 0
            for ref_text, ref_weight in external_refs:
                if ref_text and ref_text.strip():
                    external_score += ref_weight * compute_chrf_score(cand, ref_text)
                    total_ext_weight += ref_weight
            if total_ext_weight > 0:
                external_score /= total_ext_weight

        # Combined score
        if external_refs and external_weight > 0:
            combined = (1 - external_weight) * internal_score + external_weight * external_score
        else:
            combined = internal_score

        if combined > best_score:
            best_score = combined
            best_cand = cand

    return best_cand


# Quick test
test_cands = ["The silver was given to the palace.", "Silver was given to the palace.", "The gold was given."]
test_ref = [("The silver has been given to the palace.", 0.8)]
selected = consensus_select(test_cands, test_ref, 0.3)
print(f"Consensus scoring ready.")
print(f"  Test: {test_cands}")
print(f"  Selected: '{selected}'")

## TR-TRY Translation Retriever

In [ ]:
def normalize_akkadian(text):
    """Normalize Akkadian transliteration for retrieval matching."""
    if pd.isna(text) or not isinstance(text, str):
        return ''
    x = str(text).lower().strip()
    x = re.sub(r'[\u2080-\u2089\u2070-\u2079\u00b9\u00b2\u00b30-9]', '', x)
    x = re.sub(r'\{([^}]+)\}', r' DET_\1 ', x)
    x = re.sub(r'\(([a-z]{1,6})\)', r' DET_\1 ', x)
    for ch in '[]<>':
        x = x.replace(ch, ' ')
    x = re.sub(r'[\u02be\u02bf\u02c0\u02c1\']', '', x)
    x = x.replace('\u2026', ' GAP ').replace('...', ' GAP ')
    x = re.sub(r'\b[xX]{1,3}\b', ' GAP ', x)
    x = x.replace('.', ' ').replace('+', '')
    x = unicodedata.normalize('NFKD', x)
    x = ''.join(ch for ch in x if not unicodedata.combining(ch))
    x = re.sub(r'[^a-z0-9\-_ ]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def load_oracc_data(oracc_path):
    """Load ORACC parallel corpus, auto-detecting column names."""
    if not oracc_path or not os.path.exists(oracc_path):
        return pd.DataFrame(columns=['transliteration', 'translation'])

    df = pd.read_csv(oracc_path)
    print(f"  ORACC raw: {len(df)} rows, columns: {list(df.columns)}")

    # Auto-detect transliteration and translation columns
    src_col = None
    tgt_col = None
    for col in df.columns:
        cl = col.lower()
        if 'translit' in cl or 'akkadian' in cl or 'source' in cl or 'src' in cl:
            src_col = col
        elif 'translat' in cl or 'english' in cl or 'target' in cl or 'tgt' in cl:
            tgt_col = col

    if src_col is None or tgt_col is None:
        # Fallback: use first two text columns
        text_cols = [c for c in df.columns if df[c].dtype == 'object']
        if len(text_cols) >= 2:
            src_col, tgt_col = text_cols[0], text_cols[1]
            print(f"  ORACC: auto-detected columns: src='{src_col}', tgt='{tgt_col}'")
        else:
            print(f"  ORACC: could not detect columns, skipping")
            return pd.DataFrame(columns=['transliteration', 'translation'])
    else:
        print(f"  ORACC: src='{src_col}', tgt='{tgt_col}'")

    result = df[[src_col, tgt_col]].rename(
        columns={src_col: 'transliteration', tgt_col: 'translation'}
    )
    result = result.dropna(subset=['transliteration', 'translation'])
    print(f"  ORACC: {len(result)} valid pairs loaded")
    return result


class TranslationRetriever:
    """TF-IDF-based translation retrieval from training corpus."""

    def __init__(self, train_path, extra_dfs=None):
        df = pd.read_csv(train_path)
        df = df[['transliteration', 'translation']].dropna()

        # Append extra data sources (e.g., ORACC)
        if extra_dfs:
            for extra_df in extra_dfs:
                if len(extra_df) > 0:
                    df = pd.concat([df, extra_df[['transliteration', 'translation']]], ignore_index=True)

        # Deduplicate by transliteration
        df = df.drop_duplicates(subset=['transliteration'], keep='first')

        self.src_raw = df['transliteration'].tolist()
        self.tgt_raw = df['translation'].tolist()
        self.src_norm = [normalize_akkadian(s) for s in self.src_raw]

        self.char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 6))
        self.X_char = self.char_vec.fit_transform(self.src_norm)

        self.word_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
        self.X_word = self.word_vec.fit_transform(self.src_norm)

        self.src_lengths = np.array(
            [len(s.split()) for s in self.src_norm], dtype=np.float32
        )
        print(f"TR-TRY indexed {len(self.src_raw)} training pairs")

    def retrieve(self, query):
        """Retrieve best matching translation. Returns (translation, score)."""
        q_norm = normalize_akkadian(query)
        if not q_norm:
            return '', 0.0

        q_char = self.char_vec.transform([q_norm])
        q_word = self.word_vec.transform([q_norm])
        sc = (q_char @ self.X_char.T).toarray()[0]
        sw = (q_word @ self.X_word.T).toarray()[0]
        combined = W_CHAR * sc + W_WORD * sw

        k = min(TOP_K, len(self.src_norm))
        cand_idx = np.argpartition(-combined, k)[:k]
        q_len = max(1, len(q_norm.split()))
        ratio = self.src_lengths[cand_idx] / q_len
        lps = np.exp(-np.abs(np.log(ratio + 1e-5)) * LEN_PENALTY_POWER)
        final_scores = combined[cand_idx] * lps

        rerank_idx = cand_idx[np.argsort(-final_scores)[:RERANK_K]]
        best_score = -1.0
        best_idx = -1

        for idx in rerank_idx:
            seq_score = SequenceMatcher(None, q_norm, self.src_norm[idx]).ratio()
            total = final_scores[np.where(cand_idx == idx)[0][0]] + W_SEQ * seq_score
            if total > best_score:
                best_score = total
                best_idx = idx

        if best_score > MIN_ACCEPT_SCORE and best_idx >= 0:
            return self.tgt_raw[best_idx], best_score
        return '', 0.0


print("TranslationRetriever ready.")

## Load & Blend ByT5 Models

In [ ]:
def load_blended_model(model_paths, weights):
    """Load N ByT5 models and blend their weights."""
    # Filter to only models that exist
    valid = [(p, w) for p, w in zip(model_paths, weights)
             if os.path.exists(os.path.join(p, "config.json"))]
    
    if not valid:
        raise FileNotFoundError("No valid model paths found!")
    
    print(f"Blending {len(valid)} models (out of {len(model_paths)} configured)")
    
    paths, ws = zip(*valid)
    total_w = sum(ws)
    W = [w / total_w for w in ws]
    
    # Load first model as base
    print(f"  Loading base: {paths[0]} (w={W[0]:.3f})...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(paths[0])
    final_sd = base_model.state_dict()
    
    # Initialize weighted sum with base model
    for k in final_sd:
        final_sd[k] = W[0] * final_sd[k]
    
    # Add remaining models
    for i in range(1, len(paths)):
        print(f"  Loading model {i}: {paths[i]} (w={W[i]:.3f})...")
        sd = AutoModelForSeq2SeqLM.from_pretrained(paths[i]).state_dict()
        for k in final_sd:
            if k in sd:
                final_sd[k] = final_sd[k] + W[i] * sd[k]
        del sd
        gc.collect()
    
    base_model.load_state_dict(final_sd)
    gc.collect()
    
    return base_model.eval().float()


start = time.time()
model = load_blended_model(MODEL_PATHS, WEIGHTS)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
print(f"\nModel loaded and blended in {time.time() - start:.1f}s")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Build TR-TRY Index

In [ ]:
start = time.time()

# Load ORACC supplementary corpus
oracc_df = load_oracc_data(ORACC_PATH)

# Build retriever with combined data
retriever = TranslationRetriever(TRAIN_PATH, extra_dfs=[oracc_df])
print(f"TR-TRY index built in {time.time() - start:.1f}s")

## Translation Memory & OA Lexicon

Build exact-match translation memory from training data and proper noun normalizer from OA Lexicon.

In [ ]:
# === Translation Memory: exact match from training data + ORACC ===
train_df = pd.read_csv(TRAIN_PATH)
train_df = train_df.dropna(subset=['transliteration', 'translation'])

# Combine training + ORACC data for exact match
all_pairs = pd.concat([
    train_df[['transliteration', 'translation']],
    oracc_df[['transliteration', 'translation']] if len(oracc_df) > 0 else pd.DataFrame(),
], ignore_index=True).drop_duplicates(subset=['transliteration'], keep='first')

# Build exact match dict using normalized transliterations
exact_match_dict = {}
for _, row in all_pairs.iterrows():
    norm = normalize_akkadian(row['transliteration'])
    if norm and len(norm) > 3:
        exact_match_dict[norm] = str(row['translation'])

print(f"Translation memory: {len(exact_match_dict)} unique entries")
print(f"  (train: {len(train_df)}, ORACC: {len(oracc_df)})")

# === OA Lexicon: Proper Noun normalization ===
pn_ascii_to_canonical = {}  # ascii_lower -> canonical form

if os.path.exists(LEXICON_PATH):
    lex_df = pd.read_csv(LEXICON_PATH)
    pn_df = lex_df[lex_df['type'] == 'PN'].dropna(subset=['norm'])

    seen_canonical = set()
    for _, row in pn_df.iterrows():
        canonical = str(row['norm']).strip()
        if len(canonical) < 3 or canonical in seen_canonical:
            continue
        seen_canonical.add(canonical)

        # Create ASCII approximation for matching model output
        nfkd = unicodedata.normalize('NFKD', canonical)
        ascii_form = ''.join(ch for ch in nfkd if not unicodedata.combining(ch)).lower()

        if ascii_form != canonical.lower() and len(ascii_form) >= 3:
            pn_ascii_to_canonical[ascii_form] = canonical

    print(f"OA Lexicon: {len(pn_ascii_to_canonical)} PN normalization mappings")
    examples = list(pn_ascii_to_canonical.items())[:5]
    for ascii_f, canon in examples:
        print(f"  '{ascii_f}' -> '{canon}'")
else:
    print("OA Lexicon not found, skipping PN normalization")


def normalize_proper_nouns(text):
    """Replace ASCII proper noun variants with canonical diacritical forms."""
    if not pn_ascii_to_canonical or not text:
        return text
    words = text.split()
    result = []
    for word in words:
        stripped = word.strip('.,;:!?()[]"\'').lower()
        if stripped in pn_ascii_to_canonical:
            canonical = pn_ascii_to_canonical[stripped]
            if word[0].isupper():
                canonical = canonical[0].upper() + canonical[1:]
            trail = ''
            for ch in reversed(word):
                if ch in '.,;:!?()[]"\'':
                    trail = ch + trail
                else:
                    break
            result.append(canonical + trail)
        else:
            result.append(word)
    return ' '.join(result)


test_pn = normalize_proper_nouns("Ashur gave silver to Aba-ahu")
print(f"PN normalization test: '{test_pn}'")

## Adaptive Inference

In [ ]:
test_df = pd.read_csv(TEST_PATH)
n_test = len(test_df)
print(f"Test samples: {n_test}")

# Adaptive beam width based on test set size and time budget
time_budget_sec = TIME_BUDGET_MINUTES * 60

# Estimated seconds per sample for each beam width (from CPU benchmark)
beam_times = {1: 11, 2: 15, 4: 25, 8: 50}

# Choose the largest beam that fits in the time budget
chosen_beam = 1
for beam in [8, 4, 2, 1]:
    estimated_total = beam_times[beam] * n_test
    if estimated_total < time_budget_sec:
        chosen_beam = beam
        break

# Ensure beam >= num_return_sequences for multi-candidate generation
actual_return_seqs = min(NUM_RETURN_SEQUENCES, chosen_beam)
actual_beam = chosen_beam

print(f"Chosen beam width: {actual_beam}")
print(f"Return sequences: {actual_return_seqs}")
print(f"Estimated time: {beam_times[chosen_beam] * n_test / 60:.1f} min")
print(f"Time budget: {TIME_BUDGET_MINUTES} min")

GEN_PARAMS = {
    'num_beams': actual_beam,
    'num_return_sequences': actual_return_seqs,
    'max_new_tokens': 496,
    'length_penalty': 1.20,  # Tuned for chrF++ recall (v7: was 1.10)
    'early_stopping': True,
}

# Run inference
byt5_results = {}  # sample_id -> top-1 decoded string
byt5_candidates = {}  # sample_id -> list of all decoded candidates
tr_try_results = {}
exact_match_count = 0

inference_start = time.time()

with torch.inference_mode():
    for idx, row in test_df.iterrows():
        sample_id = row['id']
        raw_text = str(row['transliteration']) if pd.notna(row.get('transliteration', None)) else ''

        # 1. Check short input dictionary
        short = get_short_translation(raw_text)
        if short is not None:
            byt5_results[sample_id] = short
            byt5_candidates[sample_id] = [short]
            tr_try_results[sample_id] = (short, 1.0)
            if idx < 5:
                print(f"  [{idx}] Short input: '{raw_text}' -> '{short}'")
            continue

        # 2. Check translation memory exact match
        norm_input = normalize_akkadian(raw_text)
        if norm_input in exact_match_dict:
            exact_translation = exact_match_dict[norm_input]
            byt5_results[sample_id] = exact_translation
            byt5_candidates[sample_id] = [exact_translation]
            tr_try_results[sample_id] = (exact_translation, 1.0)
            exact_match_count += 1
            if idx < 10 or exact_match_count <= 5:
                print(f"  [{idx}] EXACT MATCH: '{raw_text[:60]}...' -> '{exact_translation[:80]}...'")
            continue

        # 3. TR-TRY retrieval
        tr_translation, tr_score = retriever.retrieve(raw_text)
        tr_try_results[sample_id] = (tr_translation, tr_score)

        # 4. ByT5 inference with multiple candidates
        input_text = preprocess_input(raw_text)
        inputs = tokenizer(
            input_text, max_length=MAX_LEN,
            padding=True, truncation=True, return_tensors='pt'
        )

        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            **GEN_PARAMS
        )

        # Decode all candidates
        n_seqs = outputs.shape[0]
        candidates = [tokenizer.decode(outputs[i], skip_special_tokens=True) for i in range(n_seqs)]
        byt5_candidates[sample_id] = candidates
        byt5_results[sample_id] = candidates[0]

        elapsed = time.time() - inference_start
        done = idx + 1
        avg_time = elapsed / max(1, done - exact_match_count)
        remaining_infer = max(0, n_test - done)
        eta = avg_time * remaining_infer

        if idx < 5 or (idx + 1) % 50 == 0:
            print(f"  [{idx}] {elapsed:.0f}s elapsed, {avg_time:.1f}s/infer, ETA {eta/60:.1f}min")
            print(f"    ByT5 ({n_seqs} cands): {candidates[0][:100]}...")
            if n_seqs > 1:
                print(f"    ByT5 alt: {candidates[1][:100]}...")
            print(f"    TR-TRY ({tr_score:.3f}): {tr_translation[:100]}...")

total_time = time.time() - inference_start
print(f"\nInference complete: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"Exact matches: {exact_match_count}/{n_test}")
print(f"ByT5 inferences: {n_test - exact_match_count}")
print(f"Candidates per inference: {actual_return_seqs}")
if n_test > exact_match_count:
    print(f"Average per ByT5 inference: {total_time/max(1, n_test - exact_match_count):.1f}s")

## Tier 2: Flan-T5 Philologist Refinement

Load Flan-T5 Philologist (250M params) after ByT5 inference.
The Philologist was trained to refine raw ByT5 translations to match gold reference style.
Sequential loading minimizes peak memory usage.

In [ ]:
# === Free ByT5 model memory before loading Philologist ===
print(f"Freeing ByT5 model memory...")
del model
del tokenizer
gc.collect()
print(f"ByT5 model freed.")

# === Tier 1 Hybrid Selection (pre-Philologist) ===
# Select best ByT5 candidate using consensus scoring before Philologist refinement
tier1_translations = {}  # sample_id -> best ByT5/TR-TRY translation
tier1_method = {}  # sample_id -> 'byt5' | 'trtry' | 'exact' | 'short'

for _, row in test_df.iterrows():
    sample_id = row['id']
    raw_text = str(row['transliteration']) if pd.notna(row.get('transliteration', None)) else ''

    # Short input override
    short = get_short_translation(raw_text)
    if short is not None:
        tier1_translations[sample_id] = short
        tier1_method[sample_id] = 'short'
        continue

    # Exact match override
    norm_input = normalize_akkadian(raw_text)
    if norm_input in exact_match_dict:
        tier1_translations[sample_id] = postprocess_translation(exact_match_dict[norm_input])
        tier1_method[sample_id] = 'exact'
        continue

    # Get ByT5 candidates and TR-TRY result
    candidates_raw = byt5_candidates.get(sample_id, [byt5_results.get(sample_id, '')])
    tr_translation, tr_score = tr_try_results.get(sample_id, ('', 0.0))

    # Postprocess all ByT5 candidates
    clean_candidates = []
    for c in candidates_raw:
        cleaned = postprocess_translation(c)
        if cleaned and not is_garbage_output(cleaned):
            clean_candidates.append(cleaned)

    tr_clean = postprocess_translation(tr_translation)

    # If all candidates are garbage, fall back to TR-TRY
    if not clean_candidates:
        tier1_translations[sample_id] = tr_clean if tr_clean else ''
        tier1_method[sample_id] = 'trtry'
        continue

    # If TR-TRY has very high confidence, prefer it
    if tr_score >= TR_TRY_HIGH_CONF and tr_clean:
        tier1_translations[sample_id] = tr_clean
        tier1_method[sample_id] = 'trtry'
        continue

    # Multi-candidate consensus scoring
    external_refs = [(tr_clean, tr_score)] if tr_clean and tr_score > 0.15 else None
    if len(clean_candidates) > 1:
        chosen = consensus_select(clean_candidates, external_refs, CONSENSUS_TRTRY_WEIGHT)
    else:
        chosen = clean_candidates[0]

    # Sentence alignment for long inputs
    if (len(raw_text) > LONG_TEXT_THRESHOLD and tr_clean
            and len(chosen) < len(tr_clean) * 0.6
            and tr_score > 0.3):
        byt5_words = chosen.split()
        tr_words = tr_clean.split()
        if len(tr_words) > len(byt5_words) + 3:
            tail_start = max(len(byt5_words) - 3, 0)
            best_pos = len(tr_words)
            best_ratio = 0
            byt5_tail = ' '.join(byt5_words[tail_start:])
            for j in range(len(tr_words) // 2, len(tr_words)):
                tr_segment = ' '.join(tr_words[max(0, j - 3):j])
                ratio = SequenceMatcher(None, byt5_tail.lower(), tr_segment.lower()).ratio()
                if ratio > best_ratio:
                    best_ratio = ratio
                    best_pos = j
            if best_ratio > 0.3 and best_pos < len(tr_words):
                chosen = chosen + ' ' + ' '.join(tr_words[best_pos:])
                chosen = chosen.strip()

    tier1_translations[sample_id] = chosen
    tier1_method[sample_id] = 'byt5'

method_counts = Counter(tier1_method.values())
print(f"\nTier 1 selection: {dict(method_counts)}")
print(f"Total: {len(tier1_translations)} translations ready for Tier 2")

# === Load Philologist model (with error handling) ===
philologist_results = {}  # sample_id -> refined translation

try:
    if PHILOLOGIST_PATH:
        print(f"\nLoading Philologist from: {PHILOLOGIST_PATH}")
        print(f"  Files: {os.listdir(PHILOLOGIST_PATH)}")
        phil_start = time.time()
        # Use Auto classes for maximum compatibility
        phil_tokenizer = AutoTokenizer.from_pretrained(PHILOLOGIST_PATH, local_files_only=True)
        phil_model = AutoModelForSeq2SeqLM.from_pretrained(PHILOLOGIST_PATH, local_files_only=True)
        phil_model.eval()
        phil_params = sum(p.numel() for p in phil_model.parameters())
        print(f"Philologist loaded in {time.time() - phil_start:.1f}s ({phil_params:,} params)")

        # === Philologist Refinement ===
        phil_infer_start = time.time()
        phil_count = 0

        with torch.inference_mode():
            for idx, row in test_df.iterrows():
                sample_id = row['id']
                tier1_text = tier1_translations.get(sample_id, '')

                # Skip short/empty translations (Philologist won't help)
                if not tier1_text or len(tier1_text.strip()) < 5:
                    continue

                # Skip exact matches (already gold-quality)
                if tier1_method.get(sample_id) == 'exact':
                    continue

                # Philologist input: prefix + raw ByT5 translation
                phil_input = PHILOLOGIST_PREFIX + tier1_text

                inputs = phil_tokenizer(
                    phil_input, max_length=384,
                    truncation=True, return_tensors='pt'
                )

                outputs = phil_model.generate(
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    **PHILOLOGIST_GEN_PARAMS
                )

                refined = phil_tokenizer.decode(outputs[0], skip_special_tokens=True)
                philologist_results[sample_id] = refined
                phil_count += 1

                if idx < 5 or (idx + 1) % 50 == 0:
                    elapsed = time.time() - phil_infer_start
                    print(f"  Phil [{idx}] {elapsed:.0f}s:")
                    print(f"    Tier 1: {tier1_text[:80]}...")
                    print(f"    Refined: {refined[:80]}...")

        phil_total = time.time() - phil_infer_start
        print(f"\nPhilologist refinement complete: {phil_total:.1f}s ({phil_count} samples)")
        if phil_count > 0:
            print(f"Average per refinement: {phil_total/phil_count:.1f}s")

        # Free Philologist memory
        del phil_model
        del phil_tokenizer
        gc.collect()
        print("Philologist model freed.")
    else:
        print("Philologist model not found, skipping Tier 2 refinement.")
except Exception as e:
    print(f"\n*** Philologist error: {e}")
    print("Falling back to Tier 1 translations only.")
    import traceback
    traceback.print_exc()
    philologist_results = {}

## Final Assembly & Postprocessing

Apply Philologist refinement results, PN normalization, and assemble final submission.

In [ ]:
final_results = []
phil_used_count = 0
phil_improved_count = 0

for _, row in test_df.iterrows():
    sample_id = row['id']
    tier1_text = tier1_translations.get(sample_id, '')
    method = tier1_method.get(sample_id, 'unknown')

    # Start with Tier 1 translation
    final_text = tier1_text

    # Apply Philologist refinement if available
    phil_text = philologist_results.get(sample_id, '')
    if phil_text and len(phil_text.strip()) > 3:
        phil_cleaned = postprocess_translation(phil_text)
        if phil_cleaned and not is_garbage_output(phil_cleaned):
            # Sanity check: Philologist output should be similar length
            # If it's drastically shorter, the refinement may have lost content
            tier1_len = len(tier1_text.split())
            phil_len = len(phil_cleaned.split())
            if phil_len >= tier1_len * 0.4 or tier1_len < 5:
                final_text = phil_cleaned
                phil_used_count += 1
                # Check if Philologist actually changed the output
                if phil_cleaned.lower().strip() != tier1_text.lower().strip():
                    phil_improved_count += 1

    # Apply PN normalization
    final_text = normalize_proper_nouns(final_text)

    # Ensure non-empty
    if not final_text or len(final_text.strip()) < 1:
        final_text = 'broken text'

    final_results.append({'id': sample_id, 'translation': final_text})

sub_df = pd.DataFrame(final_results)

print(f"\nFinal assembly:")
print(f"  Tier 1 methods: {dict(Counter(tier1_method.values()))}")
print(f"  Philologist applied: {phil_used_count} (changed: {phil_improved_count})")
print(f"  Philologist available: {len(philologist_results)}")
print(f"  Total: {len(sub_df)} samples")
print(f"  Empty translations: {(sub_df['translation'] == '').sum()}")

# Show before/after comparison for Philologist refinements
print(f"\n--- Philologist Refinement Examples ---")
shown = 0
for _, row in test_df.iterrows():
    if shown >= 5:
        break
    sid = row['id']
    t1 = tier1_translations.get(sid, '')
    phil = philologist_results.get(sid, '')
    if phil and t1 and phil.strip().lower() != t1.strip().lower():
        print(f"\n  ID {sid}:")
        print(f"    Tier 1:      {t1[:120]}...")
        print(f"    Philologist: {phil[:120]}...")
        shown += 1

sub_df.head(10)

## Save Submission

In [ ]:
sub_df.to_csv('submission.csv', index=False)
print('Saved submission.csv')
print(f'\nSample outputs:')
for _, row in sub_df.head(5).iterrows():
    print(f"  ID {row['id']}: {str(row['translation'])[:150]}")